# XTTS v2 Smoke Test (Colab)

`XTTSAdapter`(저수준 `Xtts` API 기반)가 실제 Colab GPU에서 잘 동작하는지 확인하는 단독 노트북.
학습 파이프라인과 무관하게 어댑터만 빠르게 검증한다.

검증 항목:
1. coqui-tts 설치 + XTTS v2 모델 다운로드
2. `XTTSAdapter.clone(reference_audio)` 호출
3. 출력 shape / dtype / 범위 / NaN 검증
4. 합성 결과 직접 재생
5. (옵션) `temperature` / `repetition_penalty` 다른 값 A/B 비교

런타임: **GPU (T4) 권장**. 첫 모델 다운로드에 2~3분, 합성 1회는 수 초.


## 0. GPU 확인

In [ ]:
!nvidia-smi | head -20

## 1. SDK clone + 설치

PR 머지 전이라 브랜치 `fix/xtts-low-level-api`에서 받는다.
머지 후에는 `-b main`으로 바꿔도 무방.


In [ ]:
%%bash
set -e
rm -rf voicesecure-sdk
git clone -q https://github.com/VoiceSecureHoseo/voicesecure-sdk.git
cd voicesecure-sdk
pip install -q -e .
echo "[OK] voicesecure-sdk installed"


In [ ]:
import os, sys
sys.path.insert(0, "/content/voicesecure-sdk/src")
os.chdir("/content/voicesecure-sdk")
print("cwd:", os.getcwd())


## 2. coqui-tts 설치 + XTTS v2 라이선스 동의

XTTS v2는 CPML 라이선스(연구/비상업)이므로 환경변수 `COQUI_TOS_AGREED=1` 동의 필요.
설치는 `coqui-tts` (idiap fork) 사용. 5~10분 소요.


In [ ]:
!pip install -q coqui-tts
import os
os.environ["COQUI_TOS_AGREED"] = "1"
print("[OK] coqui-tts installed, TOS agreed")


## 3. 레퍼런스 화자 WAV 준비

XTTS는 3초 이상 사람 목소리 wav 1개를 받아 그 화자로 클로닝한다.

옵션을 두 가지 제공:
- **A) 직접 업로드** — 본인이 가진 wav 파일 (16kHz 권장)
- **B) 공개 샘플 다운로드** — LibriSpeech ASR mini sample 한 토막

먼저 옵션 B를 기본으로. 자기 음성으로 바꾸려면 옵션 A 셀로 교체.

In [ ]:
# 옵션 B: 공개 영어 샘플 (한국어 ref도 가능하지만 깔끔한 공개 URL이 적어 영어로)
# XTTS는 화자 특성만 보고 합성 텍스트와 무관하게 클로닝 가능.
!wget -q -O /content/ref.wav https://github.com/coqui-ai/TTS/raw/dev/tests/data/ljspeech/wavs/LJ001-0001.wav
import os
assert os.path.exists("/content/ref.wav"), "ref.wav 다운로드 실패"
print("[OK] ref.wav:", os.path.getsize("/content/ref.wav"), "bytes")


In [ ]:
# 옵션 A (사용 시 위 셀 대신 이걸로): 직접 업로드
# from google.colab import files
# uploaded = files.upload()                  # 브라우저에서 wav 선택
# import shutil
# shutil.move(next(iter(uploaded)), "/content/ref.wav")


## 4. 16kHz mono float32로 정규화

XTTSAdapter는 내부에서 `[-1, 1]` float32 1-D를 기대한다.

In [ ]:
import numpy as np, soundfile as sf
from voicesecure.utils.audio import load_audio
ref = load_audio("/content/ref.wav")     # 16kHz mono float32 [-1, 1]
print("ref shape :", ref.shape)
print("ref dtype :", ref.dtype)
print("ref range : [%.3f, %.3f]" % (ref.min(), ref.max()))
print("ref dur   : %.2f s" % (len(ref) / 16000))
assert ref.ndim == 1 and ref.dtype == np.float32


## 5. XTTSAdapter 로드

첫 실행은 모델 가중치 (~1.8 GB)를 HuggingFace에서 다운로드한다. 2~3분 걸림.
어댑터 디폴트:
- `temperature=0.65`, `repetition_penalty=10.0`
- `top_k=50`, `top_p=0.85`
- `enable_text_splitting=True`
- `clone_text="안녕하세요. 한국어 음성 합성 테스트입니다."`
- 텍스트 전처리 `.` → `!`


In [ ]:
import time
from voicesecure.evaluators.adapters.xtts import XTTSAdapter

t0 = time.time()
adapter = XTTSAdapter()    # 모델 자동 탐색 → 캐시에 없으면 HF에서 다운로드
print(f"[load] {time.time()-t0:.1f}s elapsed")
print("model_dir :", adapter.model_dir)
print("device    :", adapter.device)
print("temp/rep  :", adapter.temperature, adapter.repetition_penalty)


## 6. clone() 호출

In [ ]:
t0 = time.time()
cloned = adapter.clone(ref)
print(f"[clone] {time.time()-t0:.1f}s elapsed")
print("cloned shape :", cloned.shape)
print("cloned dtype :", cloned.dtype)
print("cloned range : [%.3f, %.3f]" % (cloned.min(), cloned.max()))
print("cloned dur   : %.2f s" % (len(cloned) / 16000))


## 7. 자동 검증 (assert)

In [ ]:
assert isinstance(cloned, np.ndarray)
assert cloned.ndim == 1
assert cloned.dtype == np.float32
assert np.isfinite(cloned).all(), "NaN/Inf 포함"
assert cloned.min() >= -1.0 and cloned.max() <= 1.0, "범위 벗어남"
assert len(cloned) > 16000 * 0.5, f"너무 짧음: {len(cloned)/16000:.2f}s"   # 0.5초 미만이면 합성 실패 의심
print("[OK] all assertions passed")


## 8. 들어보기

In [ ]:
from IPython.display import Audio, display
print("원본 ref")
display(Audio(ref, rate=16000))
print("XTTSAdapter clone() 결과 (defaults: temp=0.65, rep=10.0)")
display(Audio(cloned, rate=16000))


## 9. (옵션) A/B 비교 — 디폴트 vs 블로그 권장 전 값

블로그가 제기한 문제(아티팩트, 반복 잡음)가 디폴트 파라미터로 얼마나 덜한지 청각 확인.

In [ ]:
# coqui 기본값 비교: temperature=0.75 (XTTS 디폴트), repetition_penalty=2.0
adapter_loose = XTTSAdapter(
    model_dir=adapter.model_dir,
    temperature=0.75,
    repetition_penalty=2.0,
    download_if_missing=False,
)
cloned_loose = adapter_loose.clone(ref)
print("loose dur  : %.2f s" % (len(cloned_loose) / 16000))
print("우리 디폴트 (temp=0.65, rep=10.0)")
display(Audio(cloned, rate=16000))
print("coqui 디폴트 (temp=0.75, rep=2.0) — 비교용")
display(Audio(cloned_loose, rate=16000))


## 10. (옵션) 합성 텍스트 바꿔보기

In [ ]:
adapter_custom = XTTSAdapter(
    model_dir=adapter.model_dir,
    clone_text="저는 보이스시큐어 SDK에서 합성된 목소리입니다.",
    download_if_missing=False,
)
out = adapter_custom.clone(ref)
display(Audio(out, rate=16000))


---

## 트러블슈팅

- **`FileNotFoundError: XTTS v2 모델 폴더를 찾을 수 없습니다`** — `download_if_missing=True` (기본)인데도 다운로드 실패. Colab 인터넷 또는 HF 토큰 문제. `!huggingface-cli login` 후 재시도.
- **`coqui-tts` 설치 충돌** — numpy/scipy 버전 충돌이면 `!pip install -q --no-deps coqui-tts` 후 부족한 deps만 따로 설치.
- **합성 길이가 0초** — `clone_text`가 비었거나 언어 코드 미스매치. `language="ko"`인지 확인.
- **GPU OOM** — `device="cpu"`로 강제 (느려짐).
